# Step 4: Visualization & Website - Chart Generation

## Overview

This notebook generates 6 interactive visualizations from Step 3 analytical tables:
- Year Distribution (Line Chart)
- Language Distribution (Pie Chart)
- Country Distribution (Horizontal Bar Chart)
- Publisher Treemap
- Publisher Linguistic Profile (Bar Chart)
- Cities Interactive Map (Folium)

All visualizations were inspired by KBR official brand colors and are exported as standalone HTML files.

## Setup and Imports

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import folium
from folium.plugins import MarkerCluster
import os
import warnings
warnings.filterwarnings('ignore')

# Configure output
pd.set_option('display.max_rows', 20)
output_dir = '../data/visualizations/'
os.makedirs(output_dir, exist_ok=True)

print("✓ Libraries imported successfully")
print(f"✓ Output directory: {output_dir}")

✓ Libraries imported successfully
✓ Output directory: ../data/visualizations/


## KBR-Inspired Brand Color Palette

In [ ]:
# KBR-Inspired Brand Colors (Reference from https://www.kbr.be/en/)
colors = {
    'dark_green': '#3D6B52',        # Primary brand color
    'navy_blue': '#1A3A5C',        # Secondary blue
    'bright_yellow': '#F4D547',    # Accent yellow
    'rose_pink': '#D4A4A0',        # Accent pink
    'warm_orange': '#E8956E',      # Accent orange
    'magenta': '#A64E7C',          # Accent magenta
    'bright_blue': '#4BA3D6',      # Accent blue
    'black': '#1A1A1A',            # Deep black
    'white': '#FFFFFF',            # White
    'light_gray': '#F5F5F5',       # Light gray
    'dark_gray': '#333333'         # Dark gray
}

# Typography settings
font_family = 'Arial, sans-serif'
title_font_size = 18
label_font_size = 12

print("✓ Color palette and typography configured")
print(f"\nKBR Color Scheme:")
for color_name, hex_code in colors.items():
    print(f"  {color_name}: {hex_code}")

✓ Color palette and typography configured

KBR Color Scheme:
  dark_green: #3D6B52
  navy_blue: #1A3A5C
  bright_yellow: #F4D547
  rose_pink: #D4A4A0
  warm_orange: #E8956E
  magenta: #A64E7C
  bright_blue: #4BA3D6
  black: #1A1A1A
  white: #FFFFFF
  light_gray: #F5F5F5
  dark_gray: #333333


## Load Analytical Data from Step 3

In [4]:
# Load analytical tables from Step 3
df_year = pd.read_csv('../data/integrated/year_analytical.csv')
df_language = pd.read_csv('../data/integrated/language_analytical.csv')
df_country = pd.read_csv('../data/integrated/country_analytical.csv')
df_publisher = pd.read_csv('../data/integrated/publisher_analytical.csv')
df_city = pd.read_csv('../data/integrated/city_map_filtered.csv')

print("=== DATA SUMMARY ===")
print(f"\nYear data: {len(df_year)} records (1602-1901)")
print(f"Language data: {len(df_language)} records, {df_language['language'].nunique()} unique languages")
print(f"Country data: {len(df_country)} records, {df_country['country_name'].nunique()} unique countries")
print(f"Publisher data: {len(df_publisher)} records, {df_publisher['publisher_name_cleaned'].nunique()} unique publishers")
print(f"City data: {len(df_city)} records, {df_city['city_name_standardized'].nunique()} unique cities")

=== DATA SUMMARY ===

Year data: 987 records (1602-1901)
Language data: 1000 records, 11 unique languages
Country data: 1014 records, 26 unique countries
Publisher data: 1024 records, 374 unique publishers
City data: 1052 records, 185 unique cities


---
# CHART 1: Year Distribution (Line Chart)

In [30]:
# Prepare data: aggregate books per year
year_dist = df_year.groupby('year').size().reset_index(name='count')
year_dist = year_dist.sort_values('year')

# Create figure
fig_year = go.Figure()

# Add line trace with area fill
fig_year.add_trace(go.Scatter(
    x=year_dist['year'],
    y=year_dist['count'],
    name='Publications per Year',
    mode='lines',
    line=dict(
        color=colors['dark_green'],
        width=3
    ),
    fill='tozeroy',
    fillcolor=f'rgba(61, 107, 82, 0.2)',  # Semi-transparent dark green
    hovertemplate='<b>Year: %{x}</b><br>Publications: %{y}<extra></extra>'
))

# Update layout
fig_year.update_layout(
    title={
        'text': 'Publication Timeline (1602-1901)',
        'font': {'size': title_font_size, 'color': colors['dark_green'], 'family': font_family},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Publication Year</br></br><i style="font-size:14px; color:#666;">Tip: Use the range slider below to zoom into specific periods</i>',
    yaxis_title='Number of Publications',
    template='plotly_white',
    hovermode='x unified',
    plot_bgcolor=colors['white'],
    paper_bgcolor=colors['white'],
    height=600,  # Increase height to accommodate range slider
    # Add range slider for interactive zooming on later period
    xaxis=dict(
        showgrid=True, 
        gridwidth=1, 
        gridcolor=colors['light_gray'],
        rangeslider=dict(visible=True, thickness=0.05),  # Enable range slider
        type='linear',
         # Set tick interval: major ticks every 50 years, minor every 10 years
        # This spreads out the dense later period more evenly on display
        dtick=50,  # Major grid lines every 50 years
        minor=dict(dtick=10)  # Minor grid lines every 10 years (optional refinement)
    ),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor=colors['light_gray']),
    font=dict(size=label_font_size, color=colors['dark_gray'], family=font_family),
    
)

# Export to HTML
output_path = f'{output_dir}year_distribution.html'
fig_year.write_html(output_path, config={'responsive': True, 'displayModeBar': True})
print(f"✓ Exported: {output_path}")
fig_year.show()

✓ Exported: ../data/visualizations/year_distribution.html


---
# CHART 2: Language Distribution (Pie Chart)

In [29]:
# Prepare data: count books by language
# Language code to full name mapping
language_names = {
    'fre': 'French',
    'ger': 'German',
    'dut': 'Dutch',
    'eng': 'English',
    'ita': 'Italian',
    'lat': 'Latin',
    'spa': 'Spanish',
    'por': 'Portuguese',
    'rus': 'Russian',
    'swe': 'Swedish',
    'multiple languages': 'Multiple languages'
}

lang_dist = df_language['language'].value_counts().reset_index()
lang_dist.columns = ['language', 'count']
lang_dist['language_full'] = lang_dist['language'].map(language_names)

# Define color mapping for languages
color_map = {
    'fre': colors['dark_green'],
    'ger': colors['navy_blue'],
    'dut': colors['rose_pink'],
    'eng': colors['warm_orange'],
    'ita': colors['magenta'],
}
other_colors = [colors['bright_blue'], colors['bright_yellow'], '#FFB6C1', '#DEB887', '#20B2AA']

chart_colors = []
for lang in lang_dist['language']:
    if lang in color_map:
        chart_colors.append(color_map[lang])
    else:
        chart_colors.append(other_colors[len(chart_colors) % len(other_colors)])

# Create pie chart
# Create pie chart with full language names
fig_lang = go.Figure(data=[go.Pie(
    labels=lang_dist['language_full'],  # Use full names instead of codes
    values=lang_dist['count'],
    marker=dict(colors=chart_colors, line=dict(color=colors['white'], width=2)),
    textinfo='label+percent',
    textfont=dict(size=14, color=colors['white'], family=font_family),
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)])

fig_lang.update_layout(
    title={
        'text': 'Linguistic Diversity of the Collection',
        'font': {'size': title_font_size, 'color': colors['dark_green'], 'family': font_family},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor=colors['white'],
    plot_bgcolor=colors['white'],
    height=600,
    font=dict(size=label_font_size, color=colors['dark_gray'], family=font_family),
    showlegend=True,
    legend=dict(x=1.0, y=1.0, xanchor='right', yanchor='top', itemclick=False, itemdoubleclick=False)
)

# Export
output_path = f'{output_dir}language_distribution.html'
fig_lang.write_html(output_path, config={'responsive': True, 'displayModeBar': True})
print(f"✓ Exported: {output_path}")
fig_lang.show()

✓ Exported: ../data/visualizations/language_distribution.html


---
# CHART 3: Country Distribution (Horizontal Bar Chart)

In [31]:
# Prepare data: count books by country
country_dist = df_country['country_name'].value_counts().reset_index()
country_dist.columns = ['country_name', 'count']
country_dist = country_dist.sort_values('count', ascending=True)

# Create color gradient
normalized_counts = (country_dist['count'] - country_dist['count'].min()) / (country_dist['count'].max() - country_dist['count'].min())
bar_colors = [f'rgba(26, 58, 92, {0.4 + 0.6 * val})' for val in normalized_counts]  # Navy blue gradient

# Highlight max value with yellow
max_idx = country_dist['count'].idxmax()
bar_colors[country_dist.index.get_loc(max_idx)] = colors['bright_yellow']

# Create horizontal bar chart
fig_country = go.Figure(data=[go.Bar(
    y=country_dist['country_name'],
    x=country_dist['count'],
    orientation='h',
    marker=dict(color=bar_colors, line=dict(color=colors['white'], width=1)),
    text=country_dist['count'],
    textposition='outside',
    textfont=dict(size=14, color=colors['dark_gray'], family=font_family),
    hovertemplate='<b>%{y}</b><br>Publications: %{x}<extra></extra>'
)])

fig_country.update_layout(
    title={
        'text': 'Geographic Scope of Publication Network',
        'font': {'size': title_font_size, 'color': colors['dark_green'], 'family': font_family},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Number of Publications',
    yaxis_title='Country',
    template='plotly_white',
    paper_bgcolor=colors['white'],
    plot_bgcolor=colors['light_gray'],
    height=600,
    hovermode='y',
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor=colors['white'], fixedrange=True),
    yaxis=dict(fixedrange=True),
    font=dict(size=label_font_size, color=colors['dark_gray'], family=font_family),
    showlegend=False
)

# Export
output_path = f'{output_dir}country_distribution.html'
fig_country.write_html(output_path, config={'responsive': True, 'displayModeBar': True})
print(f"✓ Exported: {output_path}")
fig_country.show()

✓ Exported: ../data/visualizations/country_distribution.html


---
# CHART 4: Publisher Treemap

In [27]:
import plotly.graph_objects as go
import pandas as pd
import textwrap

# ── Helper: pick white or dark-gray text based on background luminance ──
def get_text_color(hex_color):
    hex_color = hex_color.lstrip('#')
    r, g, b = int(hex_color[0:2],16), int(hex_color[2:4],16), int(hex_color[4:6],16)
    luminance = (0.299*r + 0.587*g + 0.114*b) / 255
    return '#FFFFFF' if luminance < 0.5 else '#333333'

# ── 1. Data Aggregation ──
pub_df = df_publisher[df_publisher['publisher_name_cleaned'] != 'Unknown'].copy()
pub_counts = pub_df.groupby(['period', 'publisher_name_cleaned']).size().reset_index(name='count')
pub_counts = pub_counts[pub_counts['count'] >= 2]
top_pubs = pub_counts.sort_values('count', ascending=False).head(30)

# ── 2. KBR Period Color Mapping ──
period_colors = {
    '1600-1750': '#D6D6D6',
    '1751-1800': '#4BA3D6',
    '1801-1830': '#1A3A5C',
    '1831-1860': '#D4A4A0',
    '1861-1890': '#F4D547',
    '1891-1901': '#E8956E'
}

# ── 3. Build Hierarchy (3 levels: Period → Publisher → Book) ──
hierarchy_rows = []
active_periods = top_pubs['period'].unique()

# Level 1: Period parent nodes
for p in active_periods:
    child_sum = top_pubs[top_pubs['period'] == p]['count'].sum()
    pc = period_colors.get(p, '#CCCCCC')
    hierarchy_rows.append({
        'id': p,
        'label': f"<b>{p}</b>",
        'parent': '',
        'value': child_sum,
        'color': pc,
        'textcolor': get_text_color(pc)
    })

# Build a set of valid publisher node IDs for book-level lookup
valid_pub_ids = set()

# Level 2: Publisher child nodes
for _, row in top_pubs.iterrows():
    p = row['period']
    pub = row['publisher_name_cleaned']
    pc = period_colors.get(p, '#CCCCCC')
    node_id = f"{p}|{pub}"
    valid_pub_ids.add(node_id)
    wrapped = '<br>'.join(textwrap.wrap(pub, width=15))
    label = f"{wrapped}<br>({row['count']})"
    hierarchy_rows.append({
        'id': node_id,
        'label': label,
        'parent': p,
        'value': row['count'],
        'color': pc,
        'textcolor': get_text_color(pc)
    })

# Level 3: Individual book nodes under each publisher
# Use df_publisher which already contains title, year, period, publisher_name_cleaned
pub_books = df_publisher[
    (df_publisher['publisher_name_cleaned'] != 'Unknown') &
    (df_publisher['publisher_name_cleaned'].isin(top_pubs['publisher_name_cleaned']))
].copy()

for _, row in pub_books.iterrows():
    p = row['period']
    pub = row['publisher_name_cleaned']
    parent_id = f"{p}|{pub}"

    # Only add books whose publisher is in our top_pubs hierarchy
    if parent_id not in valid_pub_ids:
        continue

    # Truncate long titles for display; preserve full title in hover
    #title_display = textwrap.shorten(str(row['title']), width=35, placeholder='…')
    year = int(row['year']) if pd.notna(row['year']) else '?'
    # Wrap full title across multiple lines (no truncation)
    wrapped_title = '<br>'.join(textwrap.wrap(str(row['title']), width=20))
    label = f"{wrapped_title}<br>({year})"

    # Use book_id to guarantee unique node IDs (titles can repeat)
    book_node_id = f"{parent_id}|{row['book_id']}"

    pc = period_colors.get(p, '#CCCCCC')
    hierarchy_rows.append({
        'id': book_node_id,
        'label': label,
        'parent': parent_id,
        'value': 1,  # Equal weight: each book counts as 1
        'color': pc,
        'textcolor': get_text_color(pc)
    })

df_tree = pd.DataFrame(hierarchy_rows)

# ── 4. Build Figure ──
fig_treemap = go.Figure(go.Treemap(
    ids=df_tree['id'],
    labels=df_tree['label'],
    parents=df_tree['parent'],
    values=df_tree['value'],
    branchvalues='total',
    marker=dict(
        colors=df_tree['color'],
        line=dict(color='#FFFFFF', width=1)
    ),
    # Show only 2 levels by default; drill into books on click
    maxdepth=2,
    textinfo='label',
    texttemplate='%{label}',
    textposition='middle center',
    textfont=dict(
        family=font_family,
        size=14,
        color=df_tree['textcolor'].tolist()
    ),
    hoverinfo='label',
    hovertemplate=None,
))

fig_treemap.update_layout(
    title={
        'text': 'Core Publisher Distribution Across Time Periods',
        'font': {'size': title_font_size, 'color': colors['dark_green'], 'family': font_family},
        'x': 0.5, 'xanchor': 'center'
    },
    margin=dict(t=100, l=0, r=0, b=0),
    paper_bgcolor='#FFFFFF',
    autosize=True,
    height=600,
    font=dict(family=font_family),
    uniformtext=dict(minsize=14, mode='hide'),
    hovermode='closest',
    annotations=[dict(
    text="<i>Tip: Click a tile to check details. To go back, click on the arrow in the top-right corner.</i>",
    x=0.5, y=1.05,
    xref='paper', yref='paper',
    xanchor='center', yanchor='bottom',
    showarrow=False,
    font=dict(size=14, color='#666666', family=font_family)
)]
)

# Export
output_path = f'{output_dir}publisher_treemap.html'
fig_treemap.write_html(output_path, config={'responsive': True, 'displayModeBar': False})
print(f"✓ Exported: {output_path}")
fig_treemap.show()

✓ Exported: ../data/visualizations/publisher_treemap.html


---
# CHART 5: Publisher Language Distribution

In [5]:
# ── Publisher Language Distribution: Stacked Bar Chart (Strict Treemap Match & Global Sort) ──

import pandas as pd
import plotly.graph_objects as go

# ==========================================
# STEP 1: EXACT REPLICATION OF YOUR TREEMAP LOGIC
# ==========================================
pub_df = df_publisher[df_publisher['publisher_name_cleaned'] != 'Unknown'].copy()

# Count books within each period to mirror the Treemap's chronological grouping
pub_counts = pub_df.groupby(['period', 'publisher_name_cleaned']).size().reset_index(name='count')

# Apply threshold: Only keep publishers with 2 or more books in a single period
pub_counts = pub_counts[pub_counts['count'] >= 2]

# Extract the exact target dataset used in your Treemap (Top 30 period-publisher segments)
top_pubs_segments = pub_counts.sort_values('count', ascending=False).head(30)

# ==========================================
# STEP 2: MERGE WITH LANGUAGE DATA & APPLY STRICT FILTER
# ==========================================
lang_df = df_language.copy()
merged = pub_df.merge(lang_df[['book_id', 'language']], on='book_id', how='inner')

if 'language_y' in merged.columns:
    merged = merged.rename(columns={'language_y': 'language'}).drop(columns=['language_x'])

# CRITICAL ACADEMIC ALIGNMENT: Keep only the books belonging to the exact (period, publisher) pairs in Treemap
top_merged = merged.merge(
    top_pubs_segments[['period', 'publisher_name_cleaned']], 
    on=['period', 'publisher_name_cleaned'], 
    how='inner'
)

# Step 3: Build cross-tabulation: publisher × language
cross = pd.crosstab(top_merged['publisher_name_cleaned'], top_merged['language'])

# ==========================================
# STEP 4: SORT LEGEND BY LANGUAGE VOLUME (ASCENDING FOR PLOTLY BOTTOM-UP STACK)
# ==========================================
# Plotly stacks legends from bottom to top, so the largest language volume 
# must be the last column in 'cross' to appear at the very top of the interactive legend.
lang_totals = cross.sum(axis=0).sort_values(ascending=True)
cross = cross[lang_totals.index]

# ==========================================
# STEP 5: PROPOSAL A - SORT PUBLISHERS BY CORE GLOBAL OUTPUT (Y-AXIS)
# ==========================================
# Sort the Y-axis publishers by their total output within this core subset.
# ascending=True is applied here because Plotly plots horizontal bars from bottom to top.
# This ensures the largest global publisher appears beautifully at the top of the chart.
cross['_total'] = cross.sum(axis=1)
cross = cross.sort_values('_total', ascending=True).drop(columns='_total') 

# Step 6: Map language codes to readable labels (Aligned with Cell 10 Pie Chart)
lang_labels = {
    'fre': 'French',
    'dut': 'Dutch',
    'ger': 'German',
    'lat': 'Latin',
    'eng': 'English',
    'spa': 'Spanish',
    'por': 'Portuguese',
    'ita': 'Italian',
    'multiple languages': 'Multiple Languages',
}

# Step 7: Language colors — strictly matching KBR brand colors (Cell 163 variable names corrected)
lang_colors = {
    'fre': colors['dark_green'],    
    'ger': colors['navy_blue'],     
    'dut': colors['rose_pink'],     
    'eng': colors['warm_orange'],   
    'ita': colors['magenta'],       
    'lat': colors['bright_blue'],   
    'spa': colors['bright_yellow'], 
    'por': '#20B2AA',               
    'multiple languages': colors['dark_gray'], 
}

# Step 8: Build stacked bar traces — traces will now be added in the sorted language order
traces = []
for lang_code in cross.columns:
    label = lang_labels.get(lang_code, lang_code.upper())
    color = lang_colors.get(lang_code, '#CCCCCC')
    traces.append(go.Bar(
        name=label,
        y=cross.index,                        # Publishers on Y-axis (horizontal bar layout)
        x=cross[lang_code],                   # Book counts on X-axis
        orientation='h',
        marker=dict(color=color, line=dict(color='#FFFFFF', width=0.5)),
        hovertemplate=f'<b>%{{y}}</b><br>Language: {label}<br>Publications: %{{x}}<extra></extra>'
    ))

# Step 9: Build figure
fig_pub_lang = go.Figure(data=traces)

fig_pub_lang.update_layout(
    barmode='stack',
    title={
        'text': 'Linguistic Profile of Core Publishers',
        'font': {'size': title_font_size, 'color': colors['dark_green'], 'family': font_family},
        'x': 0.5, 'xanchor': 'center'
    },
    xaxis=dict(
        title='Number of Publications',
        tickformat='d',
        gridcolor=colors['light_gray'],
        fixedrange=True
    ),
    yaxis=dict(
        title='Publisher (Ordered by Total Volume)',
        automargin=True,
        fixedrange=True
    ),
    legend=dict(
        title='Language (Sorted by Volume)',
        orientation='v',
        x=1.02, y=1,
        font=dict(size=14, family=font_family),
        itemclick=False,
        itemdoubleclick=False
    ),
    paper_bgcolor=colors['white'],
    plot_bgcolor=colors['white'],
    autosize=True,
    height=650,
    margin=dict(t=80, l=250, r=150, b=100),
    font=dict(size=label_font_size, color=colors['dark_gray'], family=font_family),
    
)

# Export
output_path = f'{output_dir}publisher_language_bar.html'
fig_pub_lang.write_html(output_path, config={'responsive': True, 'displayModeBar': False})
print(f"✓ Successfully exported Proposal A Chart to: {output_path}")
fig_pub_lang.show()

✓ Successfully exported Proposal A Chart to: ../data/visualizations/publisher_language_bar.html


---
# CHART 6: Cities Interactive Map (Folium)

In [170]:
# Aggregate by city
city_agg = df_city.groupby('city_name_standardized').agg({
    'latitude': 'first',
    'longitude': 'first',
    'book_id': 'count'
}).reset_index()
city_agg.columns = ['city', 'lat', 'lon', 'book_count']
city_agg = city_agg.sort_values('book_count', ascending=False)

print(f"\nTop 20 cities by publication count:")
print(city_agg.head(20))


Top 20 cities by publication count:
          city        lat        lon  book_count
128      Paris  48.856667   2.352222         296
30    Brussels  50.846667   4.351667         192
65       Ghent  51.053611   3.725278          49
95      London  51.507222  -0.127500          35
9      Antwerp  51.221111   4.399722          25
94       Liège  50.639722   5.570556          24
88     Leipzig  51.340632  12.374733          21
110       Mons  50.454722   3.952500          16
161    Tournai  50.605556   3.388056          15
114      Namur  50.466667   4.866667          13
19      Berlin  52.516667  13.383333          13
142       Rome  41.893056  12.482778          13
91       Lille  50.631944   3.057500           9
89      Leuven  50.877500   4.704444           8
64      Geneva  46.200000   6.150000           8
28      Bruges  51.208889   3.224167           7
24    Bordeaux  44.837778  -0.579444           7
156  Stuttgart  48.777500   9.180000           7
108      Milan  45.466944   9.19

In [ ]:
# Define accent color palette for circle markers (bright and eye-catching)
accent_colors = {
    'yellow': colors['bright_yellow'],    # #F4D547 - largest cities
    'magenta': colors['magenta'],         # #A64E7C - large cities
    'orange': colors['warm_orange'],      # #E8956E - medium cities
    'blue': colors['bright_blue'],        # #4BA3D6 - small cities
    'dark green': colors['dark_green']     # #3D6B52 - smallest cities
}

# Create base map with clean grayscale background (CartoDB positron)
# This ensures circle markers stand out clearly without visual clutter
map_center = [50.0, 10.0]  # Center on Europe
m = folium.Map(
    location=map_center,
    zoom_start=5,
    tiles='CartoDB positron'  # Clean grayscale background
)

# Add circle markers for each city, sized by publication count and colored by magnitude
for idx, row in city_agg.iterrows():
    city_name = row['city']
    lat, lon = row['lat'], row['lon']
    book_count = int(row['book_count'])
    
    # Scale circle radius for better visibility (was too small before)
    # Min 4px for visibility, max 30px to avoid overlap
    
    import math
    radius = max(4, min(30, math.sqrt(book_count) * 2))

    # Assign color based on publication count (creates visual hierarchy)
    if book_count >= 150:  # Largest cities (e.g., Paris)
        color_fill = accent_colors['yellow']
        color_border = colors['white']  # Dark border for contrast
    elif book_count >= 30:  # Large cities
        color_fill = accent_colors['magenta']
        color_border = colors['white']
    elif book_count >= 20:  # Medium cities
        color_fill = accent_colors['orange']
        color_border = colors['white']
    elif book_count >= 10:  # Small cities
        color_fill = accent_colors['blue']
        color_border = colors['white']
    else:  # Smallest cities
        color_fill = accent_colors['dark green']
        color_border = colors['white']
    
    # Add circle marker with improved visibility
    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        color=color_border,           # Border color (white or dark for contrast)
        fill=True,
        fillColor=color_fill,         # Fill color (bright accent colors)
        fillOpacity=0.85,
        weight=2,
        tooltip=f"<b>{city_name}</b>: {book_count} publication(s)"  # Hover tooltip
    ).add_to(m)

# Export to HTML
output_path = f'{output_dir}cities_map.html'
m.save(output_path)
print(f"✓ Exported: {output_path}")

✓ Exported: ../data/visualizations/cities_map.html


---
# Summary

In [199]:
print("\n" + "="*70)
print("STEP 4A COMPLETE: VISUALIZATION GENERATION")
print("="*70)

output_files = [
    ('year_distribution.html', 'Line Chart - Temporal Framework'),
    ('language_distribution.html', 'Pie Chart - Linguistic Diversity'),
    ('country_distribution.html', 'Bar Chart - Geographic Scope'),
    ('publisher_treemap.html', 'Treemap - Publisher Market Share'),
    ('publisher_language_bar.html', 'Stacked Bar Chart - Publisher Linguistic Profile'),
    ('cities_map.html', 'Folium Map - Geospatial Distribution')
]

print(f"\nExported visualizations ({output_dir}):")
for filename, description in output_files:
    filepath = f'{output_dir}{filename}'
    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  ✓ {filename:<40} ({size_kb:.1f} KB) - {description}")
    else:
        print(f"  ✗ {filename:<40} - NOT FOUND")

print(f"\nDesign Specifications Applied:")
print(f"  ✓ KBR brand color palette")
print(f"  ✓ Sans-serif typography (Arial)")
print(f"  ✓ Interactive hover tooltips (Plotly)")
print(f"  ✓ Responsive design for mobile/desktop")
print(f"  ✓ Standalone HTML export (no server required)")

print(f"\nNext Steps:")
print(f"  1. Review generated charts in browser")
print(f"  2. Adjust colors if needed (modify color variables above)")
print(f"  3. Write chart descriptions (200-300 words each)")
print(f"  4. Build website framework (Step 4d)")
print(f"  5. Embed charts into website pages (Step 4e)")

print(f"\n✓ Step 4a visualization generation complete")


STEP 4A COMPLETE: VISUALIZATION GENERATION

Exported visualizations (../data/visualizations/):
  ✓ year_distribution.html                   (4461.5 KB) - Line Chart - Temporal Framework
  ✓ language_distribution.html               (4460.5 KB) - Pie Chart - Linguistic Diversity
  ✓ country_distribution.html                (4461.1 KB) - Bar Chart - Geographic Scope
  ✓ publisher_treemap.html                   (4482.8 KB) - Treemap - Publisher Market Share
  ✓ publisher_language_bar.html              (4469.2 KB) - Stacked Bar Chart - Publisher Linguistic Profile
  ✓ cities_map.html                          (135.9 KB) - Folium Map - Geospatial Distribution

Design Specifications Applied:
  ✓ KBR brand color palette
  ✓ Sans-serif typography (Arial)
  ✓ Interactive hover tooltips (Plotly)
  ✓ Responsive design for mobile/desktop
  ✓ Standalone HTML export (no server required)

Next Steps:
  1. Review generated charts in browser
  2. Adjust colors if needed (modify color variables above)
  